# Dataset Preparation — SeribuCerita Emotion Classifier

**Capstone Project CC26-PSU212 — AI Path**

Notebook ini menyiapkan dataset untuk melatih model klasifikasi emosi teks Bahasa Indonesia.

### Sumber Data
| Dataset | Referensi | Jumlah | Label Asli |
|---|---|---|---|
| IndoNLU EmoT | Saputri et al., 2018 | ~4.000 tweet | anger, fear, happy, love, sadness |
| Ricco48 Public Opinion | Riccosan et al., 2022 | ~7.080 tweet | anger, fear, joy, love, sad, neutral |

### Target
5 kelas: `anger` · `fear` · `sad` · `neutral` · `happy`

### Pipeline
```
Load dari GitHub → Label Mapping → Preprocessing (BERT-friendly)
→ Deduplication → Filter Neutral → Stratified Split (80/10/10)
→ Leakage Check → Ambiguous Removal → Negation Augmentation → Save
```

### Output
- `train_5kls.csv` — 7,413 rows (termasuk augmentasi negasi)
- `valid_5kls.csv` — 931 rows
- `test_5kls.csv` — 951 rows

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TARGET_LABELS = ['anger', 'fear', 'sad', 'neutral', 'happy']
LABEL2ID = {label: idx for idx, label in enumerate(TARGET_LABELS)}

pd.set_option('display.max_colwidth', 120)
print(f'Label mapping: {LABEL2ID}')

## 2. Load Data dari GitHub

Semua dataset di-download langsung dari repository asli — tidak perlu upload manual.

In [ ]:
# IndoNLU EmoT
INDONLU_BASE = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/emot_emotion-twitter/'
INDONLU_FILES = {
    'train': INDONLU_BASE + 'train_preprocess.csv',
    'valid': INDONLU_BASE + 'valid_preprocess.csv',
    'test':  INDONLU_BASE + 'test_preprocess.csv',
}

# Ricco48
RICCO_BASE = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/'
RICCO_FILES = {
    'sad':     RICCO_BASE + 'SadData.csv',
    'fear':    RICCO_BASE + 'FearData.csv',
    'anger':   RICCO_BASE + 'AngerData.csv',
    'joy':     RICCO_BASE + 'JoyData.csv',
    'neutral': RICCO_BASE + 'NeutralData.csv',
}

In [ ]:
# Load IndoNLU EmoT (semua split digabung)
dfs_indonlu = []
for split, url in INDONLU_FILES.items():
    df = pd.read_csv(url)
    df['source'] = 'indonlu'
    dfs_indonlu.append(df)
    print(f'  IndoNLU {split}: {len(df)} rows')

df_indonlu = pd.concat(dfs_indonlu, ignore_index=True)
print(f'\nTotal IndoNLU: {len(df_indonlu)}')
print(df_indonlu['label'].value_counts())

In [ ]:
# Load Ricco48 (file berformat TSV)
dfs_ricco = []
for emotion, url in RICCO_FILES.items():
    df = pd.read_csv(url, sep='\t')
    df['source'] = 'ricco48'
    dfs_ricco.append(df)
    print(f'  Ricco {emotion}: {len(df)} rows')

df_ricco = pd.concat(dfs_ricco, ignore_index=True)
print(f'\nTotal Ricco: {len(df_ricco)}')
print(df_ricco['label'].value_counts())

## 3. Label Mapping

| Sumber | Label Asli | Target |
|---|---|---|
| IndoNLU | `sadness` | `sad` |
| IndoNLU | `love` | DROP |
| Ricco48 | `joy` | `happy` |
| Ricco48 | `love` | Tidak di-load |

In [ ]:
def process_source(df):
    """Standardize columns dan map labels ke target 5 kelas."""
    df = df.copy()
    df.columns = [c.lower() for c in df.columns]
    df['label'] = df['label'].str.lower().str.strip()
    df['label'] = df['label'].replace({'sadness': 'sad', 'joy': 'happy'})
    df = df.rename(columns={'tweet': 'text'}) if 'tweet' in df.columns else df

    before = len(df)
    df = df[df['label'].isin(TARGET_LABELS)].copy()
    print(f'  Dropped {before - len(df)} rows (label di luar target)')

    return df[['text', 'label', 'source']].reset_index(drop=True)

df_indonlu = process_source(df_indonlu)
df_ricco = process_source(df_ricco)

df_combined = pd.concat([df_indonlu, df_ricco], ignore_index=True)
print(f'\nCombined: {len(df_combined)} rows')
print(pd.crosstab(df_combined['label'], df_combined['source'], margins=True))

## 4. Text Preprocessing

Preprocessing BERT-friendly — hanya bersihkan noise, pertahankan tanda baca dan case.

| Dilakukan | Tidak dilakukan |
|---|---|
| Hapus URL, mention, placeholder IndoNLU | Case folding |
| Hapus simbol `#` (pertahankan kata) | Stopword removal |
| Kompres repetisi karakter (max 2) | Stemming / lemmatization |
| Normalize whitespace | Slang normalization |

In [ ]:
INDONLU_PLACEHOLDERS = ['[USERNAME]', '[URL]', '[SENSITIVE-NO]']

def preprocess_text(text):
    """Preprocessing minimal untuk BERT."""
    if not isinstance(text, str):
        return ''
    for ph in INDONLU_PLACEHOLDERS:
        text = text.replace(ph, ' ')
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'(.)\1{2,}', lambda m: m.group(1) * 2, text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_combined['text'] = df_combined['text'].apply(preprocess_text)

# Hapus teks yang terlalu pendek setelah cleaning
before = len(df_combined)
df_combined = df_combined[df_combined['text'].str.split().str.len() >= 2].reset_index(drop=True)
print(f'Removed {before - len(df_combined)} rows (<2 kata)')
print(f'Remaining: {len(df_combined)}')

## 5. Deduplication

Hapus duplikat untuk mencegah data leakage dan overweighting:
1. Teks sama + label beda → drop semua (ambigu)
2. Teks sama + label sama → keep first

In [ ]:
df_combined['text_lower'] = df_combined['text'].str.lower()

# Deteksi teks yang sama tapi label berbeda
conflict_texts = (
    df_combined[df_combined.duplicated(subset=['text_lower'], keep=False)]
    .groupby('text_lower')['label'].nunique()
    .pipe(lambda s: s[s > 1].index.tolist())
)
print(f'Conflicting labels: {len(conflict_texts)} texts')

before = len(df_combined)
df_combined = df_combined[~df_combined['text_lower'].isin(conflict_texts)].copy()
df_combined = df_combined.drop_duplicates(subset=['text_lower'], keep='first').reset_index(drop=True)
df_combined = df_combined.drop(columns=['text_lower'])
print(f'After dedup: {before} → {len(df_combined)}')

## 6. Filter Neutral

Kelas neutral (dari Ricco48) sering berisi teks reflektif/aforisma yang sebenarnya mengandung emosi kuat. Contoh: *"dengan bersyukur kita senantiasa diliputi rasa damai dan bahagia"* — ini seharusnya `happy`, bukan `neutral`.

Strategi: buang sample neutral yang mengandung ≥2 kata dari emotion lexicon.

In [ ]:
EMOTION_LEXICON = {
    'happy': ['bahagia', 'senang', 'gembira', 'syukur', 'bersyukur', 'damai',
              'tentram', 'indah', 'cinta', 'sayang', 'hebat', 'mantap', 'nikmat', 'sukses'],
    'sad': ['sedih', 'sepi', 'kesepian', 'kecewa', 'hancur', 'menangis', 'nangis',
            'rindu', 'kangen', 'kehilangan', 'patah', 'pilu', 'duka', 'galau', 'menyesal'],
    'anger': ['marah', 'kesal', 'benci', 'dendam', 'sebal', 'geram', 'jengkel',
              'murka', 'emosi', 'bangsat', 'anjing', 'goblok', 'tolol', 'bodoh'],
    'fear': ['takut', 'cemas', 'khawatir', 'gelisah', 'panik', 'ngeri', 'was-was', 'gugup'],
    'reflective': ['allah', 'tuhan', 'doa', 'surga', 'neraka', 'taubat', 'iman', 'rezeki'],
}
ALL_EMOTION_WORDS = {w for words in EMOTION_LEXICON.values() for w in words}

def count_emotion_words(text):
    return sum(1 for w in ALL_EMOTION_WORDS if w in text.lower())

THRESHOLD = 2
neutral_mask = df_combined['label'] == 'neutral'
neutral_df = df_combined[neutral_mask].copy()
neutral_df['emo_count'] = neutral_df['text'].apply(count_emotion_words)

clean_ids = neutral_df[neutral_df['emo_count'] < THRESHOLD].index
dropped = len(neutral_df) - len(clean_ids)

df_combined = pd.concat([
    df_combined[~neutral_mask],
    df_combined.loc[clean_ids]
], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Neutral filtered: dropped {dropped} rows')
print(f'Remaining: {len(df_combined)}')
print(df_combined['label'].value_counts())

## 7. Stratified Split (80/10/10)

In [ ]:
df_final = df_combined[['text', 'label', 'source']].rename(columns={'text': 'tweet'}).copy()
df_final['label_id'] = df_final['label'].map(LABEL2ID)

assert df_final[['tweet', 'label', 'label_id']].notna().all().all(), 'NaN detected!'

df_train, df_temp = train_test_split(
    df_final, test_size=0.2, stratify=df_final['label'], random_state=RANDOM_STATE
)
df_valid, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp['label'], random_state=RANDOM_STATE
)

print(f'Train: {len(df_train)} ({len(df_train)/len(df_final)*100:.1f}%)')
print(f'Valid: {len(df_valid)} ({len(df_valid)/len(df_final)*100:.1f}%)')
print(f'Test:  {len(df_test)} ({len(df_test)/len(df_final)*100:.1f}%)')

## 8. Post-Split Cleaning

Setelah split, lakukan:
1. **Leakage check** — pastikan tidak ada teks yang muncul di lebih dari 1 split
2. **Ambiguous removal** — hapus sample yang label-nya tidak konsisten dengan isi teks
3. **Short neutral removal** — neutral <3 kata cenderung noise

In [ ]:
# Cross-split leakage check & removal
valid_tweets = set(df_valid['tweet'])
test_tweets = set(df_test['tweet'])

before = len(df_train)
df_train = df_train[~df_train['tweet'].isin(valid_tweets | test_tweets)].reset_index(drop=True)
print(f'Leakage removed from train: {before - len(df_train)}')

# Intra-split dedup
for name, df in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    b = len(df)
    df = df.drop_duplicates(subset='tweet').reset_index(drop=True)
    if b - len(df) > 0:
        print(f'{name}: {b - len(df)} duplicates removed')
    if name == 'train': df_train = df
    elif name == 'valid': df_valid = df
    else: df_test = df

In [ ]:
# Ambiguous sample removal
ANGER_WORDS = {'marah','kesal','emosi','geram','benci','dongkol','murka','sebal','ngamuk','kesel','bete','anjir','bangsat','brengsek','tolol','bodoh','goblok'}
SAD_WORDS = {'sedih','nangis','menangis','kecewa','galau','pilu','duka','sendu','murung','nelangsa','pedih','terluka','patah hati','hancur','rindu','kangen','kehilangan'}
HAPPY_WORDS = {'senang','bahagia','gembira','cinta','sayang','happy','semangat','bangga','syukur','alhamdulillah','asyik','mantap','keren','lega','puas','ceria'}

def has_words(text, word_set):
    text_lower = text.lower()
    return any(w in text_lower for w in word_set)

def remove_ambiguous(df):
    """Hapus sample yang label-nya kontradiktif dengan isi teks."""
    mask = pd.Series(False, index=df.index)
    for idx, row in df.iterrows():
        text, label = str(row['tweet']), row['label']
        if label == 'anger' and has_words(text, SAD_WORDS) and not has_words(text, ANGER_WORDS):
            mask[idx] = True
        elif label == 'sad' and has_words(text, ANGER_WORDS) and not has_words(text, SAD_WORDS):
            mask[idx] = True
        elif label == 'happy' and has_words(text, ANGER_WORDS) and not has_words(text, HAPPY_WORDS):
            mask[idx] = True
        elif label == 'neutral' and len(text.split()) < 3:
            mask[idx] = True
    return df[~mask].reset_index(drop=True)

for name in ['train', 'valid', 'test']:
    df = {'train': df_train, 'valid': df_valid, 'test': df_test}[name]
    before = len(df)
    df = remove_ambiguous(df)
    print(f'{name}: {before - len(df)} ambiguous removed → {len(df)}')
    if name == 'train': df_train = df
    elif name == 'valid': df_valid = df
    else: df_test = df

## 9. Negation Augmentation

Tambahkan ~150 contoh kalimat negasi ke training set. Tujuan: agar model mengenali pola "tidak marah" = neutral, bukan anger.

Strategi: `negated emotion → neutral`

In [ ]:
import random
random.seed(RANDOM_STATE)

NEGATIONS = ['tidak', 'ga', 'gak', 'nggak', 'enggak', 'tak']

TEMPLATES = {
    'anger': ['aku {} marah sama dia', 'gue {} kesal sama situasi ini',
              'saya {} marah-marah terus kok', 'aku {} benci dia',
              'aku {} marah, cuma capek aja', 'saya {} emosi, cuma butuh istirahat',
              'aku {} geram, cuek aja', 'aku {} kesel sama dia sekarang',
              'aku {} dendam, sudah lupakan', 'saya {} jengkel sama kamu'],
    'sad':   ['aku {} sedih hari ini', 'gue {} nangis karena ini',
              'saya {} kecewa sama hasilnya', 'aku {} galau sekarang',
              'aku {} kehilangan semangat', 'saya {} patah hati',
              'aku {} pilu melihatnya', 'aku {} bersedih, hanya lelah saja',
              'aku {} kangen rumah sebenarnya', 'gue {} kesepian, ada teman'],
    'happy': ['aku {} bahagia sekarang', 'gue {} senang dapet kabar itu',
              'saya {} semangat hari ini', 'aku {} gembira sama hasilnya',
              'aku {} lega setelah selesai', 'saya {} bangga sama pencapaian itu',
              'aku {} merasa happy, biasa saja', 'saya {} puas dengan hasilnya',
              'aku {} girang sama kabarnya', 'saya {} ceria sebenarnya'],
    'fear':  ['aku {} takut presentasi besok', 'gue {} khawatir sama hasilnya',
              'aku {} cemas berlebihan', 'gue {} panik sama masalah ini',
              'aku {} ketakutan, santai saja', 'aku {} takut gagal sebenarnya',
              'gue {} was-was sama ujiannya', 'aku {} ngeri lihat beritanya',
              'gue {} parno sama situasinya', 'saya {} khawatir tapi oke'],
}

MANUAL = [
    'bukan marah, gue cuma capek aja', 'aku tidak marah, hanya kelelahan',
    'nggak kesal sih, biasa aja', 'ga ada yang perlu dimarahi sebenarnya',
    'bukan sedih, cuma lagi mikirin sesuatu', 'aku ga nangis kok, ada debu aja',
    'nggak kecewa sama sekali sama hasilnya', 'tidak ada yang perlu disedihkan',
    'bukan senang, gue cuma lega aja', 'nggak bahagia-bahagia amat, biasa aja',
    'bukan takut, gue cuma hati-hati aja', 'nggak khawatir sama sekali',
    'enggak cemas berlebihan kok', 'tidak takut gagal, sudah siap',
]

# Generate dari template
generated = []
for emotion, sentences in TEMPLATES.items():
    for sent in sentences:
        for neg in random.sample(NEGATIONS, 2):
            generated.append({'tweet': sent.format(neg), 'label': 'neutral'})

# Tambah manual
for text in MANUAL:
    generated.append({'tweet': text, 'label': 'neutral'})

df_aug = pd.DataFrame(generated).drop_duplicates(subset='tweet')
df_aug['label_id'] = df_aug['label'].map(LABEL2ID)

before = len(df_train)
df_train = pd.concat([df_train, df_aug], ignore_index=True)
df_train = df_train.drop_duplicates(subset='tweet').reset_index(drop=True)
print(f'Augmented: {before} → {len(df_train)} (+{len(df_train)-before} negation samples)')

## 10. Final Verification

In [ ]:
# Leakage check
train_set = set(df_train['tweet'])
valid_set = set(df_valid['tweet'])
test_set = set(df_test['tweet'])

print('=== LEAKAGE CHECK ===')
print(f'  train ∩ valid: {len(train_set & valid_set)}')
print(f'  train ∩ test:  {len(train_set & test_set)}')
print(f'  valid ∩ test:  {len(valid_set & test_set)}')

print(f'\n=== FINAL SPLITS ===')
for name, df in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    print(f'\n{name}: {len(df)} rows')
    print(df['label'].value_counts().sort_index().to_string())

## 11. Save

In [ ]:
# Drop kolom source (tidak diperlukan untuk training)
for name, df in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    cols = ['tweet', 'label', 'label_id']
    out = df[[c for c in cols if c in df.columns]]
    out.to_csv(f'../data/{name}_5kls.csv', index=False)
    print(f'Saved ../data/{name}_5kls.csv — {len(out)} rows')

print('\nDone. Files ready for training notebook.')

## Data Dictionary

| Column | Type | Description |
|---|---|---|
| `tweet` | string | Teks tweet (BERT-friendly preprocessing) |
| `label` | string | Label emosi: anger, fear, sad, neutral, happy |
| `label_id` | int | Encoded label (0=anger, 1=fear, 2=sad, 3=neutral, 4=happy) |